In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
from medmnist import PneumoniaMNIST
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

#-------------------------------------------
# SETUP DEL DISPOSITIVO
# ------------------------------------------
device=torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"dispositivo utilizzato: {device}\n")

# --------------------------------------------------------------------------
# 1.CARICAMENTO DATASET: PneumoniaMNIST(224x224 pixel) E DATA AUGMENTATION
#
#PneumoniaMNIST dataset of 5,856 pediatric chest X-Ray images
#labels: {'0': 'normal', '1': 'pneumonia'}
#Number of samples: {'train': 4708, 'val': 524, 'test': 624}
# --------------------------------------------------------------------------
trasformazione_train=transforms.Compose([ 
    transforms.RandomRotation(degrees=15), 
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.5], std=[0.5]), 
])

trasformazione_val=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
trasformazione_test=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_set=PneumoniaMNIST(split='train',transform=trasformazione_train,download=True,size=224) 
val_set=PneumoniaMNIST(split='val',transform=trasformazione_val,download=True,size=224)
test_set=PneumoniaMNIST(split='test',transform=trasformazione_test,download=True,size=224)

train_loader=DataLoader(dataset=train_set,batch_size=32,shuffle=True)
val_loader=DataLoader(dataset=val_set,batch_size=32,shuffle=False) 
test_loader=DataLoader(dataset=test_set,batch_size=1,shuffle=False) 

# ------------------------------------------
# 2. VISUALIZZAZIONE DEL TRAIN TRASFORMATO
# ------------------------------------------
dataiter=iter(train_loader)
images, labels=next(dataiter)
images_to_show=images[:8]
grid=torchvision.utils.make_grid(images_to_show, nrow=4)
grid=(grid*0.5)+0.5 
np_grid=grid.numpy()
plt.figure(figsize=(16, 8))
plt.imshow(np.transpose(np_grid, (1, 2, 0)), cmap='gray')
plt.axis('off')
plt.title("Train set trasformato:", fontsize=14, pad=15)
plt.show()

# -----------------------------------------------------------------------
# 3. ARCHITETTURA DELLA RETE NEURALE CONVOLUZIONALE 
# -----------------------------------------------------------------------
class PneumoniaCNN(nn.Module):
    def __init__(self, dropout_rate=0.4):
        super(PneumoniaCNN, self).__init__()
        #ESTRAZIONE DELLE CARATTERISTICHE
        #Blocco1: [batch,1,224,224]->[batch,32,112,112]
        self.blocco1=nn.Sequential(
            nn.Conv2d(1,32,kernel_size=5,padding=2,stride=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1)
        )
        #Blocco2: ->[batch,64,56,56]
        self.blocco2=nn.Sequential(
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        #Blocco3: ->[batch,128,28,28]
        self.blocco3=nn.Sequential(
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        #Blocco4: ->[batch,256,14,14]
        self.blocco4=nn.Sequential(
            nn.Conv2d(128,256,kernel_size=3,padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        #Blocco5: ->[batch,512,7,7]
        self.blocco5=nn.Sequential(
            nn.Conv2d(256,512,kernel_size=3,padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        #TRANSIZIONE
        self.global_pool=nn.AdaptiveAvgPool2d(1)
        #CLASSIFICATORE FINALE(FC)
        self.dropout=nn.Dropout(dropout_rate)
        #FC nascosto:[batch,512]->[batch,128]
        self.fc1=nn.Linear(512,128)
        self.attivazione_fc=nn.LeakyReLU(0.1)
        #OUTPUT:[batch,128]->[batch,1]
        self.fc_final=nn.Linear(128, 1)

    def forward(self, x):
        x=self.blocco1(x)
        x=self.blocco2(x)
        x=self.blocco3(x)
        x=self.blocco4(x)
        x=self.blocco5(x)
        x=self.global_pool(x)
        x=torch.flatten(x, 1) 
        x=self.dropout(x)
        x=self.fc1(x)
        x=self.attivazione_fc(x)
        x=self.dropout(x)
        x=self.fc_final(x)
        return x

# ------------------------------------------
# 4. INIZIALIZZAZIONE E ADDESTRAMENTO
#-------------------------------------------
modello=PneumoniaCNN(dropout_rate=0.4).to(device) 
peso_classi=torch.tensor([1214/3494], dtype=torch.float32).to(device)
criterio=nn.BCEWithLogitsLoss(pos_weight=peso_classi)

ottimizzatore=optim.Adam(
    modello.parameters(), 
    lr=1e-4, 
    weight_decay=1e-5 
)

scheduler=optim.lr_scheduler.ReduceLROnPlateau(
    ottimizzatore, 
    mode='min', 
    factor=0.5, 
    patience=2
)

epoche=100
patience_es=7

storia_loss_train, storia_acc_train, storia_auc_train=[], [], []
storia_loss_val, storia_acc_val, storia_auc_val=[], [], []

contatore=0
best_val_loss=float('inf')

print("addestramento\n")

for epoca in range(epoche):
    
    modello.train() 
    loss_train_totale, corrette_train, totale_train=0, 0, 0
    etichette_train_epoca, prob_train_epoca=[], []
    
    for immagini, etichette in train_loader:

        immagini=immagini.to(device)
        etichette=etichette.to(device).float().view(-1) 
        ottimizzatore.zero_grad()
        predizioni=modello(immagini).view(-1)

        loss=criterio(predizioni, etichette)
        loss.backward()

        ottimizzatore.step()

        loss_train_totale+=loss.item()

        probabilita=torch.sigmoid(predizioni)
        predizioni_etichette=(probabilita>0.5).float()

        corrette_train+=(predizioni_etichette==etichette).sum().item()
        totale_train+=etichette.size(0)

        etichette_train_epoca.extend(etichette.cpu().numpy())
        prob_train_epoca.extend(probabilita.detach().cpu().numpy())
        
    acc_train=corrette_train/totale_train
    loss_train_media=loss_train_totale/len(train_loader)
    
    try:
        auc_train=roc_auc_score(etichette_train_epoca, prob_train_epoca)
    except ValueError:
        auc_train=0.5 
        
    storia_loss_train.append(loss_train_media)
    storia_acc_train.append(acc_train)
    storia_auc_train.append(auc_train)

    # FASE DI VALIDAZIONE
    modello.eval()
    loss_val_totale, corrette_val, totale_val=0, 0, 0
    etichette_val_epoca, prob_val_epoca=[], []

    with torch.no_grad():
        for immagini, etichette in val_loader:

            immagini=immagini.to(device)
            etichette=etichette.to(device).float().view(-1)
            predizioni=modello(immagini).view(-1)

            loss=criterio(predizioni, etichette)
            loss_val_totale+=loss.item()

            probabilita=torch.sigmoid(predizioni)
            predizioni_etichette=(probabilita>0.5).float()
            corrette_val+=(predizioni_etichette==etichette).sum().item()
            totale_val+=etichette.size(0)
            etichette_val_epoca.extend(etichette.cpu().numpy())
            prob_val_epoca.extend(probabilita.cpu().numpy())
            
    acc_val=corrette_val/totale_val
    loss_val_media=loss_val_totale/len(val_loader)
    
    try:
        auc_val=roc_auc_score(etichette_val_epoca, prob_val_epoca)
    except ValueError:
        auc_val=0.5
        
    storia_loss_val.append(loss_val_media)
    storia_acc_val.append(acc_val)
    storia_auc_val.append(auc_val)
    scheduler.step(loss_val_media)
    lr_corrente=ottimizzatore.param_groups[0]['lr']

    print(f"Epoca {epoca+1:02d}/{epoche} | LR: {lr_corrente:.6f} | "
          f"Train Loss {loss_train_media:.4f} - ACC: {acc_train:.4f} - AUC: {auc_train:.4f} | "
          f"Val Loss {loss_val_media:.4f} - ACC: {acc_val:.4f} - AUC: {auc_val:.4f}")

    if loss_val_media<best_val_loss: 
        best_val_loss=loss_val_media 
        contatore=0 
        torch.save(modello.state_dict(), 'miglior_modello.pth') 
        print(f"nuovo miglior modello salvato. (Val Loss: {best_val_loss:.4f})") 
    else: 
        contatore+=1 
        print(f"nessun miglioramento da {contatore} epoche.")
        if contatore>=patience_es: 
            print(f"\n early stopping attivato all'epoca {epoca+1}. Addestramento interrotto.") 
            break 

# --------------------------------------------------
# 5. VALUTAZIONE SUL TEST SET E PLOT
# -------------------------------------------------
print("\n recupero i pesi del miglior modello salvato per il Test finale")
modello.load_state_dict(torch.load('miglior_modello.pth', weights_only=True))
print("modello migliore caricato con successo")

print("\n VALUTAZIONE SUL TEST SET")
modello.eval() 

tutte_le_predizioni_binarie=[]
tutte_le_probabilita=[] 
tutte_le_etichette_reali=[]

with torch.no_grad():
    for immagini, etichette in test_loader:
        
        immagini=immagini.to(device)
        etichette=etichette.to(device).float().view(-1)
        predizioni_grezze=modello(immagini).view(-1)
        probabilita=torch.sigmoid(predizioni_grezze)
        predizioni_binarie=(probabilita>0.5).float()
        tutte_le_predizioni_binarie.extend(predizioni_binarie.cpu().numpy().tolist())
        tutte_le_probabilita.extend(probabilita.cpu().numpy().tolist())
        tutte_le_etichette_reali.extend(etichette.cpu().numpy().tolist())

tutte_le_predizioni_binarie=np.array(tutte_le_predizioni_binarie)
tutte_le_probabilita=np.array(tutte_le_probabilita)
tutte_le_etichette_reali=np.array(tutte_le_etichette_reali)

cm=confusion_matrix(tutte_le_etichette_reali, tutte_le_predizioni_binarie)
TN, FP, FN, TP=cm.ravel()

accuracy=(TP+TN)/(TP+TN+FP+FN)
recall=TP/(TP+FN) if (TP+FN)>0 else 0 
specificity=TN/(TN+FP) if (TN+FP)>0 else 0
precision=TP/(TP+FP) if (TP+FP)>0 else 0 
vpn=TN/(TN+FN) if (TN+FN)>0 else 0 
f1_score=2*(precision*recall)/(precision+recall) if (precision+recall)>0 else 0
test_auc=roc_auc_score(tutte_le_etichette_reali, tutte_le_probabilita)

print(f" Accuratezza Test set: {accuracy*100:.2f}%")
print(f" Recall: {recall*100:.2f}%")
print(f" Specificity: {specificity*100:.2f}%")
print(f" Precision (VPP): {precision*100:.2f}%")
print(f" VPN: {vpn*100:.2f}%")
print(f" F1-Score: {f1_score:.4f}")
print(f" AUC (Test Set): {test_auc:.4f}")

print("\n Report dettagliato Scikit-Learn:")
print(classification_report(tutte_le_etichette_reali, tutte_le_predizioni_binarie, target_names=['Sano (0)', 'Polmonite (1)']))

print("\n Generazione dei grafici")
num_epoche=len(storia_loss_train)

plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epoche+1), storia_loss_train, label='Train Loss', color='blue', marker='o', markersize=4)
plt.plot(range(1, num_epoche+1), storia_loss_val, label='Val Loss', color='red', marker='o', markersize=4)
plt.title('Andamento della Loss media per epoca')
plt.xlabel('Epoca')
plt.ylabel('Loss media')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epoche+1), storia_acc_train, label='Train Acc', color='blue', marker='o', markersize=4)
plt.plot(range(1, num_epoche+1), storia_acc_val, label='Val Acc', color='green', marker='o', markersize=4)
plt.title("Andamento dell'Accuratezza (ACC)")
plt.xlabel('Epoca')
plt.ylabel('Accuratezza media')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epoche+1), storia_auc_train, label='Train AUC', color='purple', marker='o', markersize=4)
plt.plot(range(1, num_epoche+1), storia_auc_val, label='Val AUC', color='orange', marker='o', markersize=4)
plt.title("Andamento dell'AUC")
plt.xlabel('Epoca')
plt.ylabel('AUC')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Previsto Sano', 'Previsto Malato'], 
            yticklabels=['Vero Sano', 'Vero Malato'],
            annot_kws={"size": 14}) 
plt.title('Matrice di Confusione (sul Test Set)')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 8)) 
fpr, tpr, _=roc_curve(tutte_le_etichette_reali, tutte_le_probabilita)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {test_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasso Falsi Positivi (1 - Specificity)')
plt.ylabel('Tasso Veri Positivi (Recall)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# ----------------------------------------
# 6. EXPLAINABLE AI CON GRAD-CAM 
# ----------------------------------------
num_per_categoria=2
print(f"\n generazione delle mappe Grad-CAM. Ricerca di {num_per_categoria} immagini per ogni categoria (TP, TN, FP, FN)")

modello.eval()
target_layers=[modello.blocco5]
cam=GradCAM(model=modello, target_layers=target_layers)
targets=[ClassifierOutputTarget(0)]

categorie={
    "Vero Positivo (TP)": [],
    "Vero Negativo (TN)": [],
    "Falso Positivo (FP)": [],
    "Falso Negativo (FN)": []
}

for immagini, etichette in test_loader:
    immagini=immagini.to(device)
    etichetta_reale=int(etichette.item())
    with torch.no_grad():
        logits=modello(immagini).view(-1)
        prob_predetta=torch.sigmoid(logits).item()
        predizione_binaria=1 if prob_predetta>0.5 else 0

    if etichetta_reale==1 and predizione_binaria==1 and len(categorie["Vero Positivo (TP)"])<num_per_categoria:
        categorie["Vero Positivo (TP)"].append((immagini, etichetta_reale, prob_predetta))
    elif etichetta_reale==0 and predizione_binaria==0 and len(categorie["Vero Negativo (TN)"])<num_per_categoria:
        categorie["Vero Negativo (TN)"].append((immagini, etichetta_reale, prob_predetta))
    elif etichetta_reale==0 and predizione_binaria==1 and len(categorie["Falso Positivo (FP)"])<num_per_categoria:
        categorie["Falso Positivo (FP)"].append((immagini, etichetta_reale, prob_predetta))
    elif etichetta_reale==1 and predizione_binaria==0 and len(categorie["Falso Negativo (FN)"])<num_per_categoria:
        categorie["Falso Negativo (FN)"].append((immagini, etichetta_reale, prob_predetta))

    if all(len(lista)==num_per_categoria for lista in categorie.values()):
        break

fig, assi=plt.subplots(2, 4, figsize=(22, 11))
assi=assi.ravel()
indice_plot=0

for nome_categoria, lista_immagini in categorie.items():
    for img_tensor, etichetta_reale, prob_predetta in lista_immagini:
        grayscale_cam=cam(input_tensor=img_tensor, targets=targets, aug_smooth=True)[0, :]
        img_vis=img_tensor.squeeze().cpu().numpy()
        img_vis=(img_vis*0.5)+0.5 
        img_vis=np.clip(img_vis, 0, 1)
        img_vis_rgb=cv2.cvtColor(img_vis, cv2.COLOR_GRAY2RGB)
        cam_image=show_cam_on_image(img_vis_rgb, grayscale_cam, use_rgb=True)
        coppia_immagini=np.hstack((img_vis_rgb, cam_image/255.0))
        ax=assi[indice_plot]
        ax.imshow(coppia_immagini)
        ax.set_title(f"{nome_categoria}\nReale: {etichetta_reale} | Pred: {prob_predetta:.2f}", fontsize=11, fontweight='bold')
        ax.axis('off')
        indice_plot+=1

plt.suptitle("Analisi Comparativa Grad-CAM (Originale vs Heatmap) ordinata per Categoria Clinica", fontsize=16, y=0.98, fontweight='bold')
plt.tight_layout()
plt.show()